# OpenAI client via `jupyter_base`

This notebook shows how to call the OpenAI API through **`OpenAIClient`**, which loads the API key inside the backend package (from `.env`, a key file, or the environment) so you do not paste secrets into cells.

Prerequisites: run Jupyter with the project environment (`make run-jupyter` or the Docker setup) and configure credentials as described in the repository `README.md` and `.env.example`.

## Credentials (no key in this notebook)

Typical options:

- **`OPENAI_API_KEY`** in `.env` or in the shell environment.
- **`JUPYTER_BASE_OPENAI_KEY_FILE`** pointing to a file whose first line is the key (often under `.secrets/`, gitignored).

`load_settings()` does not copy `.env` into `os.environ`, so a key stored only in `.env` is not visible to `os.environ.get("OPENAI_API_KEY")` after `load_settings()`, while `OpenAIClient` still reads it when resolving the key.

In [1]:
import os

from jupyter_base import OpenAIClient, load_settings

# App settings (optional); does not inject OPENAI_API_KEY into the environment.
settings = load_settings()
settings.app_name

# After load_settings, a key that exists only in .env is usually absent here:
os.environ.get("OPENAI_API_KEY")

In [2]:
try:
    client = OpenAIClient()
    print("OpenAIClient ready (key resolved by jupyter_base).")
except ValueError as exc:
    client = None
    print("Not configured yet — set up credentials then re-run this cell:")
    print(exc)

OpenAIClient ready (key resolved by jupyter_base).


## Simple completion

`complete_text` sends one user message (and optional system prompt) and returns the assistant reply as a string.

In [3]:
if client is not None:
    reply = client.complete_text(
        user="Name one strength of using a small Python package as a notebook backend.",
        model="gpt-4o-mini",
        system="Answer in one short sentence.",
    )
    print(reply)
else:
    print("Skip: create a client in the previous cell after configuring credentials.")

One strength is improved performance and reduced memory usage compared to larger, more complex backends.


## Full chat completion

`chat_completion` forwards to the OpenAI SDK and returns the full response object (choices, usage, etc.).

In [4]:
if client is not None:
    response = client.chat_completion(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You reply with a single word: yes or no."},
            {"role": "user", "content": "Is JupyterLab a web application?"},
        ],
    )
    choice = response.choices[0]
    print(choice.message.content)
    if response.usage is not None:
        print("token usage:", response.usage.model_dump())
else:
    print("Skip: no client.")

Yes.
token usage: {'completion_tokens': 2, 'prompt_tokens': 30, 'total_tokens': 32, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}


## Optional: keep `OPENAI_API_KEY` out of the kernel

To start Jupyter with `OPENAI_API_KEY` unset in the server and kernel (while still using a key file), run from the repo:

```bash
make run-jupyter JUPYTER_STRIP_OPENAI_ENV=1
```

Point `JUPYTER_BASE_OPENAI_KEY_FILE` at your key file in `.env`. Notebook code remains ordinary Python, so this reduces accidental exposure via `os.environ`, not a full sandbox.